In [ ]:
# mount Google drive
from google.colab import drive
# load the liabries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
# This will list the files in your specific folder
drive.mount('/content/drive')
# the list of airports we restirc ourselves to
airport_limit_list =["JFK","LAX","MIA","SFO","EWR","ORD","ATL","DFW","IAH",
"BOS","MCO","FLL","SEA","CLT","DEN","PHL","LAS","HNL","DTW","MSP","PHX","LGA","TPA",
"SLC","BWI","AUS","SAN","HOU","PDX","MDW","OAK","BNA","DCA","STL","DAL"]
# the airlines are already filterd to the ones only that we use.
source = "/content/drive/MyDrive/Datamining/Feature_ingeneered_Data/" # where we load the Data from
destination = "/content/drive/MyDrive/Datamining/Data_for_Model/" #where to put the Data, that is ready to be used in an ML Model.
output_file_name = "preprocessed.feather"
source_file = "feature_ingeneared.feather"

probabliy you cshoudl delete most of the code startign from here, to have a clean notebook for preprocessing

In [ ]:

df = pd.read_feather(source + source_file)
display(df)

In [ ]:
# show for each column the number of unique values, missing values and the number of mssing values when rows with Divred and Cannceld flights are dropped, as these flights have a lot of missing values in the arrival delay column
for col in df.columns:
    unique_values = df[col].nunique()
    missing_values = df[col].isna().sum()
    if missing_values > 0:
        print(f"{col}:  Unique:   {unique_values} ---  Missing: {missing_values}")

In [ ]:
# drop unessary columns
# Timezone information, as we have already merged the weather data and know the timezone of the departure and arrival airports
# timestamps that are not schedulad departure and sheduled arrival
# other things that are not useful for training
orther_cols =['CRSDepDateTime_UTC']

df = df.drop(columns=orther_cols)
# convert sheduled serv
df ['CRSDepDateTime'] = df['CRSDepDateTime'].dt.hour * 60 + df['CRSDepDateTime'].dt.minute
df ['CRSArrDateTime'] = df['CRSArrDateTime'].dt.hour * 60 + df['CRSArrDateTime'].dt.minute
display(df)

In [ ]:
# list of categrcal columns to be One-Hot Encoded
# cat_One_Hot =['Reporting_Airline','type_DEP','type_ARR','most_common_surface_DEP','most_common_surface_ARR']
cat_One_Hot =['Reporting_Airline','Origin','Dest']

# scale the numeric columns, except for the target column
scal_params = ['CRSDepDateTime', 'CRSArrDateTime', 'temp','prcp','wspd','temp_ARR','prcp_ARR','wspd_ARR','TurnaroundTime','CRSElapsedTime','Distance']

continus_cols =[]
# convert CRSDepDateTime and CRSArrDateTime to minutes since midnight


target = 'ArrDelayMinutes'
# split the Data into train and test set


In [ ]:
# save the Data to do

In [ ]:
# One Hot Encode the categorical columns
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler
encoder = OneHotEncoder(handle_unknown="ignore")
encoded_carriers = encoder.fit_transform(df[cat_One_Hot])
encoded_carriers_df = pd.DataFrame(encoded_carriers.toarray(), columns=encoder.get_feature_names_out(cat_One_Hot))
# join the dataframes together
df = pd.concat([df.drop(columns=cat_One_Hot), encoded_carriers_df], axis=1, join="inner")


# scale the numeric columns
scaler = MinMaxScaler()
df[scal_params] = scaler.fit_transform(df[scal_params])

In [ ]:
# perform a time based split of the data, using the CRSDepDateTime column, to split the data into a train set with flights before 2023-01-01 and a test set with flights after 2023-01-01
test_threshold = pd.to_datetime("2011-08-01").timestamp()
train = df[df['CRSDepDateTime'] < test_threshold]
test = df[df['CRSDepDateTime'] >= test_threshold]
X_train = train.drop(columns=[target])
y_train = train[target]
X_test = test.drop(columns=[target])
y_test = test[target]
print(f"Length of train set: {len(train)}")
print(f"Length of test set: {len(test)}")


In [ ]:
# perform a time sensitve hyperparmeter tuning and cross validation





In [ ]:
# function for performing a time sensitve hyperparmeter tuning and cross validation, using the TimeSeriesSplit and RandomizedSearchCV from sklearn, with a given model and parameter grid
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
def time_sensitive_hyperparameter_tuning(model, param_grid, X_train, y_train):
    splitter = TimeSeriesSplit(n_splits=5)
    random_search = RandomizedSearchCV(model, param_grid, cv=splitter, n_iter=10, scoring='R2', n_jobs=-1)
    random_search.fit(X_train, y_train)
    return random_search.best_estimator_, random_search.best_params_, random_search.best_score_

In [ ]:
# test the model
def test_model(model, X_test, y_test):
    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    print(f"Mean Squared Error: {mse}")
    print(f"R^2 Score: {r2}")
    return mse, r2

# fucntion to plot Predicted vs Actual values
def plot_predictions(Y_test, predictions, title=f"Predicted vs Actual {target}", Metrics:dict=None):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=Y_test, y=predictions, alpha=0.5)
    plt.plot([Y_test.min(), Y_test.max()], [Y_test.min(), Y_test.max()], 'r--')  # Line for perfect predictions
    plt.xlabel(f"Actual {target}")
    plt.ylabel(f"Predicted {target}")
    # cut off the axis at 0 and 450 to focus on the most relevant range of values
    plt.xlim(0, 450)
    plt.ylim(0, 450)
    plt.title(title)
    if Metrics:
        plt.text(0.05, 0.95, f"MSE: {Metrics['MSE']:.2f}\nR²: {Metrics['R2']:.2f}", transform=plt.gca().transAxes, verticalalignment='top')
    plt.show()

In [ ]:
# pipline
def pipeline(model, param_grid, X_train, y_train, X_test, y_test):
    best_model, best_params, best_score = time_sensitive_hyperparameter_tuning(model, param_grid, X_train, y_train)
    print(f"Best Hyperparameters: {best_params}")
    print(f"Best Cross-Validation R² Score: {best_score:.4f}")
    mse, r2 = test_model(best_model, X_test, y_test)
    plot_predictions(y_test, best_model.predict(X_test), title=f"Predicted vs Actual {target} with Best Hyperparameters", Metrics={"MSE": mse, "R2": r2})

def test_on_train_subset(model, X_train_subset, Y_train_subset):
    predictions = model.fit(X_train_subset, Y_train_subset).predict(X_train_subset)
    mse = mean_squared_error(Y_train_subset, predictions)
    r2 = r2_score(Y_train_subset, predictions)
    print(f"Train Subset - Mean Squared Error: {mse:.2f}")
    print(f"Train Subset - R^2 Score: {r2:.2f}")
    return model, predictions, mse, r2

In [ ]:
# function for tree or forest models to plot the feature importance
def plot_feature_importance(model, feature_names, top_n=50):
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1][:top_n]
    plt.figure(figsize=(10, 10))
    sns.barplot(x=importances[indices], y=np.array(feature_names)[indices])
    plt.title("Feature Importances")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.show()

In [ ]:
# try out a simple linear regression model
from sklearn.linear_model import LinearRegression
model = LinearRegression()
param_grid = {
    'fit_intercept': [True, False],
    'normalize': [True, False]
}
pipeline(model, param_grid, X_train, y_train, X_test, y_test)

In [ ]:
# Descion Tree Regressor
from sklearn.tree import DecisionTreeRegressor
model = DecisionTreeRegressor(random_state=42, min_samples_leaf=50)
param_grid = {
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
pipeline(model, param_grid, X_train, y_train, X_test, y_test)

In [ ]:
# XGBoost Regressor
from xgboost import XGBRegressor
model = XGBRegressor(random_state=42, n_estimators=100, max_depth=25, learning_rate=0.1)
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30],
    'learning_rate': [0.01, 0.1, 0.2]
}
pipeline(model, param_grid, X_train, y_train, X_test, y_test)
# find the 30 most important features for the XGBoost model
plot_feature_importance(model, X_train.columns)


In [ ]:
# adaboost regressor
from sklearn.ensemble import AdaBoostRegressor
model = AdaBoostRegressor(random_state=42, n_estimators=100, learning_rate=0.1)
pipline(model)
# plot the feature importance of the adaboost model, top 50 features
plot_feature_importance(model, X_train2.columns)

In [ ]:
# random forest regressor
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=42, n_estimators=100, max_depth=25)
pipline(model)
plot_feature_importance(model, X_train2.columns)
